# 00. 사전점검

시연 전에 CW-Lite, target, 펌웨어, 측정 신호를 확인합니다. 이 노트북은 invalid, `h`, `h0px3` 세 입력을 25 dB에서 짧게 측정합니다. 정상 파형이 확인되면 01번부터 차례로 진행합니다.

## 장치와 펌웨어 확인

CW-Lite가 정확히 한 대 연결되어 있어야 합니다. prebuilt 비밀번호 펌웨어의 SHA-256도 함께 출력해 시연에 사용할 입력을 확인합니다.

In [1]:
import hashlib
import importlib
import json

import chipwhisperer as cw

from cw_sca_tutorial.demo import repository_root
import cw_sca_tutorial.hardware as hardware

hardware = importlib.reload(hardware)
diagnose_password_signal = hardware.diagnose_password_signal

ROOT = repository_root()
devices = cw.list_devices()
assert len(devices) == 1, f"Expected one ChipWhisperer, found: {devices}"

firmware = ROOT / "firmware/prebuilt/basic-passwdcheck-CWLITEARM.hex"
firmware_sha256 = hashlib.sha256(firmware.read_bytes()).hexdigest()
print("Device   :", devices[0])
print("Firmware :", firmware.name)
print("SHA-256  :", firmware_sha256)

Device   : {'name': 'ChipWhisperer-Lite', 'sn': '50203220594a48303130373132323034', 'hw_loc': (1, 4)}
Firmware : basic-passwdcheck-CWLITEARM.hex
SHA-256  : d76a5d321a24fcd7c70bcc316bc06e47eed6ddddec6320f0a2b21dc35ae90ccd


## 세 입력으로 신호 확인

target 플래시를 지우고 번들 비밀번호 펌웨어를 기록합니다. invalid와 `h`는 `PASSWORD FAIL`, `h0px3`는 `Access granted`를 반환해야 합니다. 25 dB 파형 폭이 50 ADC count보다 작거나 clipping이 있으면 다음 노트북으로 넘어가지 않습니다.

In [2]:
run = diagnose_password_signal(
    firmware,
    gains_db=(25.0,),
    repeats=1,
    runs_root=ROOT / "runs",
    serial_number=devices[0]["sn"],
    platform_name="CWLITEARM",
    target_identification="STM32F303",
)

metadata = json.loads((run / "metadata.json").read_text())
summary = metadata["gain_summaries"][0]
print("Run          :", run.name)
print("UART         :", metadata["uart_validation"])
print("Peak-to-peak :", summary["median_peak_to_peak"])
print("ADC range    :", summary["minimum"], summary["maximum"])

assert metadata["state"] == "complete"
assert metadata["uart_validation"] == "passed"
assert metadata["accepted_trace_count"] == 3
assert summary["median_peak_to_peak"] >= 50
assert summary["minimum"] > 0 and summary["maximum"] < 1023
print("Pre-class checks passed.")

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24445                     to 5000                     
scope.adc.trig_count                     changed from 1083737                   to 23927676                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538471                 
scope.clock.adc_rate                     changed from 0.0                       to 29538471.0               
scope.clock.clkgen_

## 점검이 실패하면

CW-Lite USB, 20-pin cable, target 전원, `MEASURE` SMA와 shunt 경로를 차례로 확인합니다. 이 장치의 정상 파형 폭은 약 600 ADC count였습니다. 35 dB 이상으로 gain을 올려 통과시키지 마십시오. 이전 측정에서는 ADC clipping이 발생했습니다.